# 010. 수강생 실습 - 토크나이저 4종 비교

## 학습 목표
똑같은 문장을 서로 다른 토크나이저에 넣으면 결과가 어떻게 달라지는지 직접 관찰합니다.
특히 **영어와 한국어를 처리하는 방식의 차이**, 그리고 **사전에 없는 단어(OOV)** 를
각 토크나이저가 어떻게 다루는지에 주목하세요.

| 토크나이저 | 방식 | 특징 |
|-----------|------|------|
| Keras Tokenizer | rule-based (공백·구두점 분리) | 영어에 적합, 한국어에 취약 |
| KoNLPy Okt | 사전 기반 형태소 분석 | 한국어 형태소 단위, 미등록어에 취약 |
| SentencePiece | subword (데이터로 학습) | 사전 없이 학습, OOV 가 없음 |
| tiktoken | BPE (byte 단위) | GPT 계열 모델이 사용 |

In [ ]:
# 실습에 필요한 패키지 설치 (Colab 기준 — 로컬에 이미 설치돼 있으면 생략 가능)
!pip install -q konlpy sentencepiece tiktoken

In [ ]:
# 실습용 예시 문장 — 모든 과제에서 공통으로 사용합니다.
sentences_E = [
    'I love my dog',
    'I love my cat',
    'You love my dog!',
    'I was born in Korea and graduaged University in USA.',
]

sentences_K = [
    "코로나가 심하다",
    "코비드-19가 심하다",
    '아버지가방에들어가신다',
    '아버지가 방에 들어가신다',
    '너무너무너무는 나카무라세이코가 불러 크게 히트한 노래입니다',
]

---
## 과제 1. Keras Tokenizer 로 영어 문장 토큰화하기

Keras 의 `Tokenizer` 는 **공백과 구두점을 기준으로** 단어를 나누는 rule-based 토크나이저입니다.

**할 일**:
- `Tokenizer(num_words=100, oov_token='<OOV>')` 로 토크나이저 객체를 만드세요.
- `fit_on_texts(sentences_E)` 로 영어 문장에 대해 단어 사전을 구축하세요.
- `word_index` 를 출력해 어떤 단어가 어떤 정수에 매핑됐는지 확인하세요.

**힌트**: `oov_token` 은 학습에 없던 단어(Out-Of-Vocabulary)를 대체할 특수 토큰입니다.

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer

# 빈도수 상위 100개 단어로 구성된 토크나이저 객체 생성
tokenizer = Tokenizer(num_words=100, oov_token='<OOV>')

# 영어 문장 리스트로 단어 사전 구축
tokenizer.fit_on_texts(sentences_E)

# 구축된 단어 인덱스 사전 확인
word_index = tokenizer.word_index
print(word_index)

**관찰 포인트**
- 영어는 단어 사이 띄어쓰기가 명확해 공백 분리만으로도 토큰화가 깔끔하게 됩니다.
- 모든 단어가 **소문자로 변환**되고 구두점(`!`, `.`)은 제거됩니다.
- `<OOV>` 가 인덱스 1번을 차지합니다 — 학습에 없던 단어는 모두 이 토큰으로 처리됩니다.

---
## 과제 2. 같은 Keras Tokenizer 로 한국어 문장 토큰화하기

영어에서 잘 동작했던 rule-based 토크나이저를 그대로 한국어에 적용해 봅니다.

**할 일**:
- 새 `Tokenizer` 객체를 만들고 `fit_on_texts(sentences_K)` 로 한국어 사전을 구축하세요.
- `word_index` 를 출력하고, **과제 1의 영어 결과와 무엇이 다른지** 비교하세요.

In [ ]:
tokenizer = Tokenizer(num_words=100, oov_token='<OOV>')

# 한국어 문장 리스트로 단어 사전 구축
tokenizer.fit_on_texts(sentences_K)

vocabulary_keras_korean = tokenizer.word_index
print(vocabulary_keras_korean)

**관찰 포인트 — rule-based 토크나이저의 한국어 한계**
- '코로나가' 와 '코비드-19가' 처럼 **명사 + 조사** 가 통째로 한 단어가 됩니다.
  같은 명사라도 조사가 바뀌면 전혀 다른 단어로 인식됩니다.
- '아버지가방에들어가신다' 는 띄어쓰기가 없어 **문장 전체가 한 단어**가 됩니다.
- → 한국어는 조사·어미가 붙는 **교착어**라 단순 공백 분리로는 의미 단위를 잡을 수 없습니다.

---
## 과제 3. KoNLPy Okt 형태소 분석기로 한국어 토큰화하기

Okt 는 한국어 **사전을 기반으로** 문장을 형태소(의미를 가진 최소 단위)로 분리합니다.

**할 일**:
- `Okt` 객체를 만들고, 각 한국어 문장에 `okt.morphs(문장)` 를 적용해 형태소 리스트를 출력하세요.
- 마지막 문장은 `okt.pos(문장)` 로 품사 태깅 결과도 함께 확인하세요.

In [ ]:
from konlpy.tag import Okt

# Okt 형태소 분석기 객체 생성
okt = Okt()

# 각 한국어 문장을 형태소 단위로 분리
for sent in sentences_K:
    print(sent)
    print("  →", okt.morphs(sent), "\n")

In [ ]:
# 사전 미등록어 확인 — 품사 태깅 결과 관찰
print(okt.pos('너무너무너무는 나카무라세이코가 불러 크게 히트한 노래입니다'))

**관찰 포인트 — 형태소 분석의 장점과 OOV 문제**
- Okt 는 '코로나/가', '방/에' 처럼 **명사와 조사를 분리**합니다.
  덕분에 '코로나', '방' 같은 명사를 조사와 무관하게 일관되게 추출할 수 있습니다.
- 그러나 '너무너무너무', '나카무라세이코' 는 **Okt 사전에 없는 단어**라
  여러 개의 엉뚱한 조각으로 쪼개집니다 → **사전 기반 방식의 OOV 문제**.
- 신조어·고유명사가 많은 실제 텍스트에서는 사용자 사전을 직접 추가해야 합니다.

---
## 과제 4. SentencePiece 토크나이저 직접 학습시키기

SentencePiece 는 **사전 없이** 말뭉치(corpus)로부터 자주 등장하는 글자 조각(subword)을
스스로 학습합니다. 여기서는 네이버 영화 리뷰(NSMC) 약 15만 문장으로 학습합니다.

**할 일**:
- NSMC 학습 데이터를 내려받아 리뷰 문장만 `nsmc.txt` 파일로 저장하세요.
- `spm.SentencePieceTrainer.Train(...)` 으로 vocab 3만 개짜리 모델을 학습하세요.

**힌트**: `pandas.read_csv` 에서 `quoting=3` 은 따옴표를 특수문자로 처리하지 않는 옵션입니다.

In [ ]:
import tensorflow as tf
import pandas as pd
import sentencepiece as spm

# NSMC(네이버 영화 리뷰) 학습 데이터 다운로드 (캐시됨)
DATA_TRAIN_PATH = tf.keras.utils.get_file(
    "ratings_train.txt",
    "https://github.com/ironmanciti/infran_NLP/raw/main/data/naver_movie/ratings_train.txt")

# 데이터 로드 후 결측치 제거
train_data = pd.read_csv(DATA_TRAIN_PATH, sep='\t', quoting=3)
train_data.dropna(inplace=True)

# 리뷰 문장(document 열)만 txt 파일로 저장 — SentencePiece 학습 입력
with open('./nsmc.txt', 'w', encoding='utf-8') as f:
    for line in train_data.document.values:
        try:
            f.write(line + '\n')
        except:
            print("write error ---> ", line)

print("학습용 문장 수:", len(train_data))

In [ ]:
# SentencePiece 모델 학습 (입력 파일 / 모델 접두사 / 어휘 사전 크기 지정)
spm.SentencePieceTrainer.Train(
    '--input=nsmc.txt --model_prefix=nsmc --vocab_size=30000')

# 학습된 모델 로드
sp = spm.SentencePieceProcessor()
sp.Load('nsmc.model')

print("SentencePiece 학습 완료 — vocab 크기:", sp.get_piece_size())

---
## 과제 5. SentencePiece 로 한국어 문장 토큰화하기

학습된 SentencePiece 모델로 예시 한국어 문장을 토큰화합니다.

**할 일**:
- `sp.encode_as_pieces(문장)` 으로 subword 토큰을, `sp.encode_as_ids(문장)` 으로 정수 ID 를 출력하세요.
- 과제 3의 Okt 결과(특히 '코비드-19', '너무너무너무')와 비교하세요.

**힌트**: 토큰 앞의 `▁` 기호는 "여기서부터 새 단어가 시작된다"는 표시입니다.

In [ ]:
for line in sentences_K:
    pieces = sp.encode_as_pieces(line)   # subword 토큰
    ids = sp.encode_as_ids(line)         # 정수 ID 시퀀스
    print(line)
    print("  토큰:", pieces)
    print("  ID  :", ids, "\n")

**관찰 포인트 — subword 방식의 강점**
- '코비드-19' 같은 신조어도 작은 조각의 조합으로 표현되므로
  **OOV(미등록어)가 원리적으로 발생하지 않습니다**.
- 자주 등장하는 표현은 하나의 토큰으로, 드문 표현은 잘게 쪼개집니다.
- 사람이 만든 사전이 필요 없고, 학습 말뭉치만 있으면 어떤 언어든 적용할 수 있습니다.

---
## 과제 6. tiktoken (BPE) 으로 영어와 한국어 비교하기

tiktoken 은 OpenAI 가 만든 BPE(Byte Pair Encoding) 토크나이저로,
GPT 계열 모델이 사용하는 것과 동일한 방식입니다.

**할 일**:
- `tiktoken.get_encoding("cl100k_base")` 로 인코더를 만드세요.
- 영어 문장과 한국어 문장을 각각 `encode` 하고 **토큰 개수**를 비교하세요.

In [ ]:
import tiktoken

# cl100k_base : GPT-4 / GPT-3.5-turbo 가 사용하는 인코딩
encoding = tiktoken.get_encoding("cl100k_base")

print("=== 영어 문장 ===")
for s in sentences_E:
    tokens = encoding.encode(s)
    print(f"{s}")
    print(f"  토큰 수: {len(tokens)}, ID: {tokens}\n")

print("=== 한국어 문장 ===")
for s in sentences_K:
    tokens = encoding.encode(s)
    print(f"{s}")
    print(f"  토큰 수: {len(tokens)}, ID: {tokens}\n")

**관찰 포인트 — 언어별 토큰 효율**
- 영어는 적은 토큰 수로 효율적으로 인코딩됩니다.
- 한국어는 byte 단위로 잘게 쪼개져 **글자 수 대비 토큰 수가 영어보다 훨씬 많습니다**.
- GPT API 비용은 토큰 수에 비례하므로, 같은 의미라도 한국어 처리 비용이 더 큽니다.

---
## 종합 정리

같은 문장을 4가지 토크나이저로 처리하며 각 방식의 원리와 한계를 관찰했습니다.

| 토크나이저 | 분리 단위 | 사전 필요? | OOV 처리 | 한국어 적합도 |
|-----------|----------|-----------|----------|--------------|
| Keras Tokenizer | 공백·구두점 | 불필요 | `<OOV>` 로 치환 | 낮음 (조사·띄어쓰기 의존) |
| KoNLPy Okt | 형태소 | 필요 (한국어 사전) | 잘못 분리됨 | 높음 (단, 미등록어 취약) |
| SentencePiece | subword | 불필요 (데이터로 학습) | 발생하지 않음 | 높음 |
| tiktoken | BPE (byte) | 불필요 (사전학습됨) | 발생하지 않음 | 가능 (토큰 수 많음) |

**핵심 메시지**: 토크나이저는 "정답"이 하나가 아니라 **언어 특성과 목적에 따라 선택**하는 도구입니다.
현대 LLM 들이 SentencePiece·BPE 같은 subword 방식을 쓰는 이유는 **OOV 가 없으면서 사전도 필요 없기** 때문입니다.

---
## 추가 실습 (선택 과제)

1. 자신만의 한국어 문장(신조어·이모지 포함)을 `sentences_K` 에 추가하고
   4가지 토크나이저 결과가 어떻게 달라지는지 비교하세요.
2. SentencePiece 의 `vocab_size` 를 `1000` 으로 줄여 다시 학습한 뒤,
   토큰이 더 잘게 쪼개지는지 관찰하세요.
3. tiktoken 으로 같은 의미의 한국어 문장과 영어 문장의 토큰 수를 비교해
   몇 배 차이가 나는지 계산해 보세요.
4. `sp.decode_ids(...)` 와 `encoding.decode(...)` 로 ID 시퀀스를 원문으로 복원해 보세요.